# This script takes a json file as input and prints it out in dictionary structure to use on the test_invoke notebook.

In [5]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, Union


PathLike = Union[str, Path]

## Load a JSON file and return the parsed object (expected to be a top-level dict).

In [6]:
def load_json(json_path: PathLike) -> Dict[str, Any]:
    path = Path(json_path)
    data = json.loads(path.read_text(encoding="utf-8"))
    if not isinstance(data, dict):
        raise ValueError(f"Expected top-level JSON object to be a dict, got {type(data).__name__}")
    return data

###    Convert a Python object (from parsed JSON) into a nicely formatted Python literal.
###   Uses pprint so booleans become True/False (not true/false).

### Return a copy/paste-ready Python assignment, e.g.
###        PARAMETERS = {...}

In [7]:
def _to_python_literals(s: str) -> str:
    # Convert JSON literals -> Python literals
    s = re.sub(r"\btrue\b", "True", s)
    s = re.sub(r"\bfalse\b", "False", s)
    s = re.sub(r"\bnull\b", "None", s)
    return s


def _is_simple_scalar(x: Any) -> bool:
    return x is None or isinstance(x, (str, int, float, bool))


def _format_inline_list(lst: list[Any]) -> str | None:
    """
    Return a one-line JSON representation for lst if it is "simple" enough.
    Otherwise return None.
    """
    # Only inline lists of scalars (numbers/strings/bools/None)
    if not all(_is_simple_scalar(x) for x in lst):
        return None
    # Use json.dumps so strings get double-quotes and floats get JSON formatting
    return json.dumps(lst, separators=(", ", ": "))


def pretty_format(
    obj: Any,
    *,
    indent: int = 4,
    level: int = 0,
    max_inline_list_items: int = 6,
    max_inline_list_width: int = 60,
    sort_keys: bool = False,
) -> str:
    """
    Pretty format a JSON-like object into a JSON-style string, but:
      - Dicts are multi-line with indentation
      - Lists that are "simple" + short are kept on one line, e.g. [100, 1, 1]
      - Other lists are expanded multi-line
    """
    pad = " " * (indent * level)
    pad_in = " " * (indent * (level + 1))

    if isinstance(obj, dict):
        items = list(obj.items())
        if sort_keys:
            items.sort(key=lambda kv: kv[0])

        if not items:
            return "{}"

        parts = ["{"]
        for i, (k, v) in enumerate(items):
            key = json.dumps(k)  # ensures double quotes
            val = pretty_format(
                v,
                indent=indent,
                level=level + 1,
                max_inline_list_items=max_inline_list_items,
                max_inline_list_width=max_inline_list_width,
                sort_keys=sort_keys,
            )
            comma = "," if i < len(items) - 1 else ""
            parts.append(f"{pad_in}{key}: {val}{comma}")
        parts.append(f"{pad}}}")
        return "\n".join(parts)

    if isinstance(obj, list):
        # Try inline list formatting
        if len(obj) <= max_inline_list_items:
            inline = _format_inline_list(obj)
            if inline is not None and len(inline) <= max_inline_list_width:
                return inline

        if not obj:
            return "[]"

        parts = ["["]
        for i, item in enumerate(obj):
            val = pretty_format(
                item,
                indent=indent,
                level=level + 1,
                max_inline_list_items=max_inline_list_items,
                max_inline_list_width=max_inline_list_width,
                sort_keys=sort_keys,
            )
            comma = "," if i < len(obj) - 1 else ""
            parts.append(f"{pad_in}{val}{comma}")
        parts.append(f"{pad}]")
        return "\n".join(parts)

    # Scalars
    return json.dumps(obj)


def dict_to_parameters_text(
    data: Dict[str, Any],
    *,
    var_name: str = "PARAMETERS",
    add_type_hint: bool = True,
    indent: int = 4,
    sort_keys: bool = False,
    max_inline_list_items: int = 6,
    max_inline_list_width: int = 60,
) -> str:
    body = pretty_format(
        data,
        indent=indent,
        level=0,
        max_inline_list_items=max_inline_list_items,
        max_inline_list_width=max_inline_list_width,
        sort_keys=sort_keys,
    )
    body = _to_python_literals(body)

    prefix = f"{var_name}: Dict[str, Any] = " if add_type_hint else f"{var_name} = "
    return prefix + body


def print_parameters_from_json(
    json_source: Union[PathLike, Dict[str, Any]],
    *,
    var_name: str = "PARAMETERS",
    add_type_hint: bool = True,
    indent: int = 4,
    sort_keys: bool = False,
    max_inline_list_items: int = 6,
    max_inline_list_width: int = 60,
) -> None:
    data = load_json(json_source) if not isinstance(json_source, dict) else json_source
    text = dict_to_parameters_text(
        data,
        var_name=var_name,
        add_type_hint=add_type_hint,
        indent=indent,
        sort_keys=sort_keys,
        max_inline_list_items=max_inline_list_items,
        max_inline_list_width=max_inline_list_width,
    )
    print(text)

### Option A: from a JSON file path


### Option B: if you already loaded it as a dict
#### cfg = load_json("Example1.json")
#### print_parameters_from_json(cfg)

In [8]:
print_parameters_from_json("Example1.json", max_inline_list_items=6, max_inline_list_width=80)

PARAMETERS: Dict[str, Any] = {
    "Simulation": {
        "simulation_type": "FullCascade",
        "screening_type": "ZBL",
        "electronic_stopping": "SRIM13",
        "electronic_straggling": "Off",
        "nrt_calculation": "NRT_element",
        "intra_cascade_recombination": False,
        "time_ordered_cascades": True,
        "correlated_recombination": True,
        "move_recoil": False,
        "recoil_sub_ed": False
    },
    "Transport": {
        "flight_path_type": "Variable",
        "flight_path_const": 1.0,
        "min_energy": 1.0,
        "min_recoil_energy": 1.0,
        "min_scattering_angle": 2.0,
        "max_rel_eloss": 0.05,
        "mfp_range": [1.0, 1e+30]
    },
    "IonBeam": {
        "ion": {
            "symbol": "H",
            "atomic_number": 1,
            "atomic_mass": 1.00784
        },
        "energy_distribution": {
            "type": "SingleValue",
            "center": 1000000.0,
            "fwhm": 1.0
        },
        "spatial_d